In [17]:
import os
import sys
import subprocess
import re
from rdkit import Chem
from rdkit.Chem import rdFMCS
from rdkit.Chem import rdchem
from rdkit.Chem import Draw
from rdkit.Chem import AllChem
from rdkit.Chem import rdDepictor
from rdkit.Chem.Draw import rdMolDraw2D
import numpy as np
from copy import deepcopy
from openbabel import pybel
import pandas as pd



# This script processes all .mol2 files in a specified directory using the cgenff tool

mol2_dir = "/home/raheelx/cphmd_walkthrough/mol2_Epik"
output_dir = "/home/raheelx/cphmd_walkthrough/cgenff_output"
os.makedirs(output_dir, exist_ok=True)

for filename in sorted(os.listdir(mol2_dir)):
    if filename.endswith(".mol2"):
        mol2_path = os.path.join(mol2_dir, filename)
        basename = filename.replace(".mol2", "")
        output_str = os.path.join(output_dir, f"{basename}.str")
        cmd = f"module load cgenff && cgenff -a < {mol2_path} > {output_str}"
        try:
            subprocess.run(cmd, shell=True, executable="/bin/bash", check=True)
            print(f"Processed {filename}")
        except subprocess.CalledProcessError as e:
            print(f"Failed processing {filename}: {e}")



CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_1.mol2
Processed riboflavin_2.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_3.mol2
Processed riboflavin_4.mol2


CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...
attype warning: enolate not explicitly supported
CHARMM General Force Field (CGenFF) program version 3.0
released May 2023
Copyright (C) 2020 SilcBio LLC
and University of Maryland, School of Pharmacy. All Rights Reserved.

Now processing molecule Riboflav ...


Processed riboflavin_5.mol2
Processed riboflavin_6.mol2


In [6]:
# Defining MCS function

### Requirments for successful use:
###     ** Molecules MUST be spacially aligned
###     ** Atom names (within each mol2 file) must be unique for each atom
###        (atom names can be similar between different mol2 files)
###     ** Atom names cannot have the "+" symbol in their name



def MsldMCS(molfile,mcsout,cutoff=0.8,debug=False):
    """
    Use bond-connectivity, atomtype definitions, and distance metrics to identify the
    core and fragment atoms for a set of supplied molecules.

    ** Molecules should be in a mol2 format; Atomtype parameters should be in a CHARMM RTF format
    ** Molecules must be spacially aligned (such that common core atoms are within 1.0A of each other)
    ** Atom names must be unique for each atom per molecule; cross molecule atom name similarities are ok
    ** Atom names cannot have the "+" character in their name

    """

    ms=[]    # ms is a list of mol2 files
    atoms=[] # atoms is a list of lists of all atom names
    heavy=[] # heavy is a list of lists of HEAVY atom names only
    xyzs=[]  # xyzs is a list of dictionaries for coordinates
    bonds=[] # bonds is a list of np.arrays of bonds (0/1 = no/yes bond)
    types=[] # types is a list of dictionaries of atom names -> atom types
    
    ## (1) Read "mol_list"
    fp=open(molfile,'r')
    for line in fp:
        ms.append(line.rstrip())
    fp.close()
    
    ## (2) Read each mol file to load in the coordinates for each atom and its bonds
    # loop over each file
    for m in range(len(ms)):
        atoms.append([])
        heavy.append([])
        xyzs.append({})
        fp=open(ms[m]+'.mol2','r')
        line=fp.readline()
        while line:
            if line.rstrip() == '@<TRIPOS>ATOM':
                # extract atom names and coordinates
                line=fp.readline()
                while line and line.rstrip() not in ('@<TRIPOS>BOND', '@<TRIPOS>SUBSTRUCTURE>'):
                    if line.strip() == "":
                        line = fp.readline()
                        continue
                    tmp = line.split()
                    if len(tmp) <6:
                        line = fp.readline()
                        continue
                    atoms[m].append(tmp[1])
                    xyzs[m][tmp[1]] = {
                    'X': float(tmp[2]),
                    'Y': float(tmp[3]),
                    'Z': float(tmp[4]),
                    'Elem': tmp[5].split('.')[0]
                }
                    if tmp[5].split('.')[0] != 'H':
                        heavy[m].append(tmp[1])
                    line = fp.readline()
                    
            if line.rstrip() == '@<TRIPOS>BOND':
                bonds.append(np.zeros((len(atoms[m]),len(atoms[m])),dtype=int))
                line=fp.readline()
                # create a bond matrix
                while line:
                    if line == '\n':
                        # empty line
                        break
                    if line[0] == '@':
                        # signifies end of "BOND" section of mol2
                        break
                    tmp=line.split()
                    bonds[m][int(tmp[1])-1][int(tmp[2])-1]=1
                    bonds[m][int(tmp[2])-1][int(tmp[1])-1]=1
                    line=fp.readline()
            line=fp.readline()
        
        fp.close()
    
    # #debug#
    # if debug:
    #     for m in range(len(ms)):
    #         print("Molecule",ms[m])
    #         print(atoms[m])
    #         print(xyzs[m])
    #         #print(bonds[m])
    #         for l in range(bonds[m].shape[0]):
    #             print(bonds[m][l])
    #         print("")
    #         print("")
    # #debug#

    ## (3) Read the str file to get the atom types
    for m in range(len(ms)):
        types.append({})
        numLPs=0
        #fp=open(ms[m]+'.str','r')  # msld_chk splits str files into rtf/prm files
        fp=open(ms[m]+'.rtf','r')
        line=fp.readline()
        while line:
            if line[0:4] == 'ATOM':
                tmp=line.split()
                # check for LonePair site
                if tmp[1][0:2] == 'LP':
                    numLPs+=1
                else:
                    types[m][tmp[1]] = tmp[2]
            line=fp.readline()
        # chk for same number of atoms
        if len(types[m]) != len(atoms[m]):
            print("ERROR: Inconsistent # of atoms between types[m] and atoms[m] after reading RTF file",ms[m]+'.str')
            print("len(types[m])=",len(types[m]),"; len(atoms[m])=",len(atoms[m]))
            quit()
    
    # #debug#
    # if debug:
    #     for m in range(len(types)):
    #         print(types[m])
    #         print("")
    # #debug#
    
    
    ## (4) Start finding the core (heavy atom only search)
    # function to create an atom's bond pattern 
    def getBonded(atomnum,molnum):
        # first pull out the atom indices that "atomnum" is bonded to
        b1=[]
        for at in range(bonds[molnum].shape[0]):
            if bonds[molnum][atomnum][at] == 1:
                b1.append(at)
        # now convert those indices to atom types
        b2=[]
        for at in range(len(atoms[molnum])):
            if at in b1:
                b2.append(types[molnum][atoms[molnum][at]])
        # make the dictionary current atomtype -> bonding neighbors
        bonded=['1','2']
        bonded[0]=types[molnum][atoms[molnum][atomnum]]  # bonded[0] = current atom's type
        bonded[1]=b2                                     # bonded[1] = neighbor types
    
        return bonded
    
    
    ## (i) Find all atom type matches
    matches=[]
    for m1 in range(len(ms)):
        matches.append({})
        for at1 in range(len(atoms[m1])):
            # compare only heavy atoms
            if xyzs[m1][atoms[m1][at1]]['Elem'] == 'H':
                continue
            matches[-1][atoms[m1][at1]]=[]
            patt1=getBonded(at1,m1)
            for m2 in range(len(ms)):
                matches[-1][atoms[m1][at1]].append([])
                if m2 == m1:
                    continue
                for at2 in range(len(atoms[m2])):
                    if xyzs[m2][atoms[m2][at2]]['Elem'] == 'H':
                        continue
                    patt2=getBonded(at2,m2)
                    # compare patterns 1 and 2 - store if they match
                    # if at1 and at2 types match and they both have the same number of neighbors
                    if (patt1[0] == patt2[0]) and (len(patt1[1]) == len(patt2[1])):
                        atsum=0
                        for at in patt1[1]:
                            if at in patt2[1]:
                                atsum+=1
                                # remove at from patt2[1] to prevent duplicate matches
                                for i in range(len(patt2[1])):
                                    if at == patt2[1][i]:
                                        patt2[1].pop(i)
                                        break
                        if atsum == len(patt1[1]):
                            # store possible match
                            matches[m1][atoms[m1][at1]][m2].append(atoms[m2][at2])
    
    # debug functions
    def printListDict(struct):
        for m in range(len(ms)):
            print("Molecule",m)
            keys = struct[m].keys()
            for k in keys:
                print(k,struct[m][k])
            print("")
    def printListList(struct):
        for m in range(len(ms)):
            print("Molecule",m)
            for l in struct[m]:
                print(l)
            print("")
    
    if debug:
        print("VERY BEGINNING")
        printListDict(matches)
    
                
    ## (ii) start sorting through the matches
    ## (iiA) remove entries with more than one empty list 
    for m1 in range(len(ms)):
        droplist=[]
        ibuff=-1
        for at1 in range(len(atoms[m1])):
            empty=0
            if xyzs[m1][atoms[m1][at1]]['Elem'] == 'H':
                continue
            ibuff+=1
            for m2 in range(len(ms)):
                if matches[m1][atoms[m1][at1]][m2] == []:
                    empty+=1
            if empty > 1:
                droplist.append(atoms[m1][at1])
        # pop off items from matches dict
        for i in droplist:
            matches[m1].pop(i)
    
    # debug
    # if debug:
    #     printListDict(matches)
                
    ## (iiB) remove entries with exactly one match and add them to cores
    cores=[] # core atoms and their matches
    for m1 in range(len(ms)):
        droplist=[]
        cores.append({})
        for at1 in range(len(atoms[m1])):
            if atoms[m1][at1] in matches[m1].keys():
                chk=0
                for m2 in range(len(ms)):
                    if m1 != m2:
                        if len(matches[m1][atoms[m1][at1]][m2]) == 1:
                            chk+=1
                if chk == len(ms) - 1: # then we have a core atom match
                    droplist.append(atoms[m1][at1])
        for i in droplist:
            at=matches[m1].pop(i)
            cores[m1][i]=at
    
    # # debug
    if debug:
        print("##AFTER QUICK EMPTY or 1 MATCH CHECKS")
        print("##CORES")
        printListDict(cores)
        print("")
        print("##MATCHES")
        printListDict(matches)
        print("")
        print("")

    
    ## (iiC) Make sure all core lists have the same number of atoms: Correct inconsistencies
    chkcores=[]
    for m1 in range(len(ms)):
        chkcores.append([])
        m1cores=list(cores[m1].keys())
        for m2 in range(len(ms)):
            chkcores[m1].append([])
            if m1 != m2:
                m2cores=list(cores[m2].keys())
                # generate list of m1 core matches from cores[m2] key-values
                m2matches=[]
                for at2 in m2cores:
                    m2matches.append(cores[m2][at2][m1][0])
                for at1 in m1cores:
                    if not (at1 in m2matches):
                        chkcores[m1][m2].append(at1)
                        # at1 = key in m1
                        line=deepcopy(cores[m1][at1])
                        line[m1]=[at1]
                        cores[m2][line[m2][0]]=line
                        cores[m2][line[m2][0]][m2]=[]
    
    ## (iiD) for remaining atoms, try to match by common bonds to core atoms
    for m1 in range(len(ms)):
        for at1 in range(len(atoms[m1])):
            m1list=[]
            if atoms[m1][at1] in matches[m1].keys():
                # identify bonded core atoms
                for at2 in range(len(atoms[m1])):
                    if atoms[m1][at1] == atoms[m1][at2]:
                        continue
                    if atoms[m1][at2] in cores[m1].keys():
                        if bonds[m1][at1][at2] == 1: # meaning there's a bond
                            m1list.append(atoms[m1][at2])
                m2list=[]
                for m2 in range(len(ms)):
                    m2list.append([])
                    if m1 != m2:
                        for at2 in range(len(atoms[m2])):
                            if atoms[m2][at2] in matches[m1][atoms[m1][at1]][m2]:
                                m2list[-1].append([])
                                # identify bonded core atoms
                                for at3 in range(len(atoms[m2])):
                                    if atoms[m2][at2] == atoms[m2][at3]:
                                        continue
                                    if atoms[m2][at3] in cores[m2].keys():
                                        if bonds[m2][at2][at3] == 1: # there's a bond
                                            m2list[-1][-1].append(atoms[m2][at3])
                # Now start comparing core-bondedness similarities
                droplist=[]
                for m2 in range(len(ms)):
                    droplist.append([])
                    if m1 != m2:
                        ibuff=-1
                        for at2 in range(len(atoms[m2])):
                            if atoms[m2][at2] in matches[m1][atoms[m1][at1]][m2]:
                                ibuff+=1
                                if len(m2list[m2][ibuff]) != len(m1list):
                                    droplist[-1].append(atoms[m2][at2])
                                    continue
                                chk=0
                                for c1 in range(len(m1list)):
                                    mcomp=cores[m1][m1list[c1]][m2][0]  # this is the core atom in m2 that matches m1list[c1]
                                    if mcomp in m2list[m2][ibuff]:
                                        chk+=1
                                if chk != len(m1list):
                                    droplist[-1].append(atoms[m2][at2])
                                    continue
    
                #if debug:
                #    print(atoms[m1][at1],m1list)
                #    print(m2list)
                #    print('droplist',droplist,'\n')
    
                # Now rebuild matches, while removing the droplist items (matches[molecule][key] = list of list)
                tmplist = []
                for m2 in range(len(ms)):
                    tmplist.append([])
                    if m1 != m2:
                        for at2 in range(len(atoms[m2])):
                            if atoms[m2][at2] in matches[m1][atoms[m1][at1]][m2]:
                                if not (atoms[m2][at2] in droplist[m2]):
                                    tmplist[m2].append(atoms[m2][at2])
                # interpret tmplist for potential removal from matches
                empty=0
                chk=0
                for m2 in range(len(ms)):
                    if tmplist[m2] == []:
                        empty+=1
                    elif len(tmplist[m2]) == 1:
                        chk+=1
                    else:
                        pass
                if empty > 1:
                    matches[m1].pop(atoms[m1][at1])
                elif chk == len(ms)-1:
                    matches[m1].pop(atoms[m1][at1])
                    cores[m1][atoms[m1][at1]]=deepcopy(tmplist)
                    # need to update cores/matches of other molecules at the same time
                    fixlist=deepcopy(tmplist)
                    fixlist[m1]=[atoms[m1][at1]]
                    for mfix in range(len(ms)):
                        if mfix == m1: continue # m1 matches/cores lists already fixed above
                        matfix=fixlist[mfix][0] # the atom to pop off matches and add to cores
                        # skip molecules with mismatched number of core atoms; errors corrected downstream
                        if not (matfix in matches[mfix].keys()): continue
                        mfixlist=deepcopy(fixlist) # create a tmplist called mfixlist
                        mfixlist[mfix]=[]
                        matches[mfix].pop(matfix)  # effect the updates
                        cores[mfix][matfix]=deepcopy(mfixlist)
                else:
                    matches[m1][atoms[m1][at1]]=deepcopy(tmplist)
    
    # # debug
    if debug:
        print("##BEFORE CORE CHK")
        print("##MATCHES")
        printListDict(matches)
        print("")
        print("##CORES")
        printListDict(cores)
        print("")
        print("")
    
    ## (iiE) Double check that new two (or more) core atoms do not have the same match. If they do,
    ## then move them out of cores and back into matches. Let next step (distances) try to fix things
    for m1 in range(len(ms)):
        m1keys=list(cores[m1].keys())
        for m2 in range(len(ms)):
            if m1 != m2:
                droplist=[]
                for k1 in range(len(m1keys)):
                    at=cores[m1][m1keys[k1]][m2][0]
                    for k2 in range(k1+1,len(m1keys)):
                        if at == cores[m1][m1keys[k2]][m2][0]:
                            # then we have a match that we need to fix
                            if not (m1keys[k1] in droplist):
                                droplist.append(m1keys[k1])
                            if not (m1keys[k2] in droplist):
                                droplist.append(m1keys[k2])
                if len(droplist) > 0:
                    for k in droplist:
                        matches[m1][k]=cores[m1][k]
                        cores[m1].pop(k)
                    m1keys=list(cores[m1].keys())
               
              
    
    # # debug
    if debug:
        print("AFTER CORE CHK, BUT BEFORE DIST CHECKS")
        print("")
        printListDict(matches)
        print("")
        print("")
        printListDict(cores)
        print("")
        print("")
    
    
    ## (iiF) RMSD analysis between remaining match atoms (last resort)
    ## *** !!! This requires the molecules to be ALIGNED to work correctly !!! ***
    ## To check that all molecules are suitably aligned, do RMSD of corrent CORE atoms
    ## (exit with error message if RMSD is > cutoff)
    
    
    #cutoff = 0.8    # disregard atom pairs with RMSDs above this value ## uncomment to hardcode this
    
       
    # For each core atom, compute an "average" xyz position, and then calc the RMSD for all
    # real atomic coordinates to this average "reference" position
    rmsd={}
    for m1 in range(1):   # treat first molecule as the "reference molecule"
        for k in cores[m1].keys():
            avg={"X":0.00,"Y":0.00,"Z":0.00}
            rmsd[k]=0.00
            # calc avg position first
            avg['X']+=xyzs[m1][k]["X"]
            avg['Y']+=xyzs[m1][k]["Y"]
            avg['Z']+=xyzs[m1][k]["Z"]
            for m2 in range(1,len(ms)):
                avg['X']+=xyzs[m2][cores[m1][k][m2][0]]["X"]
                avg['Y']+=xyzs[m2][cores[m1][k][m2][0]]["Y"]
                avg['Z']+=xyzs[m2][cores[m1][k][m2][0]]["Z"]
            avg['X']=avg['X']/float(len(ms))
            avg['Y']=avg['Y']/float(len(ms))
            avg['Z']=avg['Z']/float(len(ms))
    
            # then calc rmsd for all atoms pairs to this average position
            tmp=xyzs[m1][k]["X"]-avg['X']
            rmsd[k]+=tmp*tmp
            tmp=xyzs[m1][k]["Y"]-avg['Y']
            rmsd[k]+=tmp*tmp
            tmp=xyzs[m1][k]["Z"]-avg['Z']
            rmsd[k]+=tmp*tmp
            for m2 in range(1,len(ms)):
                tmp=xyzs[m2][cores[m1][k][m2][0]]["X"]-avg['X']
                rmsd[k]+=tmp*tmp
                tmp=xyzs[m2][cores[m1][k][m2][0]]["Y"]-avg['Y']
                rmsd[k]+=tmp*tmp
                tmp=xyzs[m2][cores[m1][k][m2][0]]["Z"]-avg['Z']
                rmsd[k]+=tmp*tmp
            rmsd[k]=np.sqrt(rmsd[k]/float(len(ms)))
    
    chk=0
    for k in rmsd.keys():
        if rmsd[k] > cutoff:
            chk+=1
            print("Large RMSD for CORE atom",k,"- RMSD >",cutoff,"("+str(round(rmsd[k],3))+").")
    if chk == len(cores[0].keys()) and len(cores[0].keys()) != 0:
        print("ERROR: Large RMSDs detected for ALL core atoms.\n     ",
              "This suggests that the molecules are not properly aligned!. Check and resubmit")
        quit()
    
    ## (iiG) Assuming the rmsd calculation has successfully completed and there were no alignment problems
    ## Move on to using distance to try to deduce if the remaining atoms pairs can be matched and moved into the core
    ## Calc distance between atom pairs and eliminate options above the cutoff value (cutoff+0.3)
    
    for m1 in range(len(ms)):
        for at1 in range(len(atoms[m1])):
            if atoms[m1][at1] in matches[m1].keys():
                droplist=[]
                for m2 in range(len(ms)):
                    droplist.append([])
                    if m1 != m2:
                        for at2 in range(len(atoms[m2])):
                            if atoms[m2][at2] in matches[m1][atoms[m1][at1]][m2]:
                                dist=0.00
                                tmp=xyzs[m1][atoms[m1][at1]]['X']-xyzs[m2][atoms[m2][at2]]['X']
                                dist+=tmp*tmp
                                tmp=xyzs[m1][atoms[m1][at1]]['Y']-xyzs[m2][atoms[m2][at2]]['Y']
                                dist+=tmp*tmp
                                tmp=xyzs[m1][atoms[m1][at1]]['Z']-xyzs[m2][atoms[m2][at2]]['Z']
                                dist+=tmp*tmp
                                dist=np.sqrt(dist)
                                if dist > cutoff+0.3:
                                    droplist[-1].append(atoms[m2][at2])
    
                # Now rebuild matches, while removing the droplist items (matches[molecule][key] = list of list)
                tmplist = []
                for m2 in range(len(ms)):
                    tmplist.append([])
                    if m1 != m2:
                        for at2 in range(len(atoms[m2])):
                            if atoms[m2][at2] in matches[m1][atoms[m1][at1]][m2]:
                                if not (atoms[m2][at2] in droplist[m2]):
                                    tmplist[m2].append(atoms[m2][at2])
                # interpret tmplist for potential removal from matches
                empty=0
                chk=0
                for m2 in range(len(ms)):
                    if tmplist[m2] == []:
                        empty+=1
                    elif len(tmplist[m2]) == 1:
                        chk+=1
                    else:
                        pass
                if empty > 1:
                    matches[m1].pop(atoms[m1][at1])
                elif chk == len(ms)-1:
                    matches[m1].pop(atoms[m1][at1])
                    cores[m1][atoms[m1][at1]]=deepcopy(tmplist)
                else:
                    matches[m1][atoms[m1][at1]]=deepcopy(tmplist)
    
    
    # debug
    if debug:
        print("AFTER DIST CHKS")
        print("MATCHES")
        printListDict(matches)
        print("")
        print("CORE ATOMS")
        printListDict(cores)
        print("")
        print("")
    
    
    ## (iiH) Try a variety of checks to ensure that all atoms in matches are either (i) classified
    ## as a core atom or discarded as a future fragment atom
    def emptyMatches(mollist):
        """ check if the matches structure is empty for all molecules in mollist """
        echk=0
        for mol in range(len(mollist)):
            if len(matches[m]) != 0:
                echk+=1
        return echk
    
    # skip checks if matches is already empty
    chk=emptyMatches(ms)
    if chk == 0:
        pass
    else:
        # for remaining atoms, search their matches for atoms that might have been identified as a core atom already;
        # if found, remove those atoms from matches[m][at]; classify matches[m][at] as core or discarded atom if possible
        for m1 in range(len(ms)):
            for at1 in range(len(atoms[m1])):
                if atoms[m1][at1] in matches[m1].keys():
                    droplist=[]
                    for m2 in range(len(ms)):
                        droplist.append([])
                        if m1 != m2:
                            # append the index of at1 in matches[m1][atoms[m1][at1]][m2]
                            for at2 in range(len(matches[m1][atoms[m1][at1]][m2])):
                                if matches[m1][atoms[m1][at1]][m2][at2] in cores[m2].keys():
                                    droplist[m2].append(at2)
                            # remove droplist
                            droplist[m2].sort(reverse=True)
                            for d in droplist[m2]:
                                matches[m1][atoms[m1][at1]][m2].pop(d)
                    # check for easy to remove match lines
                    empty=0
                    chk=0
                    for m2 in range(len(ms)):
                        if matches[m1][atoms[m1][at1]][m2] == []:
                            empty+=1
                        elif len(tmplist[m2]) == 1:
                            chk+=1
                        else:
                            pass
                    if empty > 1:
                        matches[m1].pop(atoms[m1][at1])
                    elif chk == len(ms)-1:
                        cores[m1][atoms[m1][at1]]=matches[m1].pop(atoms[m1][at1])
                    else:
                        pass
        
    
    ## (iiI) Make sure all atoms are out of matches - exit with error if not
    chk=emptyMatches(ms)
    if chk > 0:
        print("ERROR: Atoms have not been classified as core or fragment atoms!\n    ",
              "User input is required to continue running. Please specify the matching atom(s)",
              "for each atom in each molecule\n")
        for m1 in range(len(ms)):
            if len(matches[m1]) > 0:
                print("### Molecule",m1,"("+ms[m1]+")")
                print(matches[m1])
            for at1 in range(len(atoms[m1])):
                if atoms[m1][at1] in matches[m1].keys():
                    print("Molecule",m1,"("+ms[m1]+") - ATOM",atoms[m1][at1],"matches with:")
                    for m2 in range(len(ms)):
                        if m1 != m2:
                            if len(matches[m1][atoms[m1][at1]][m2]) > 1:
                                useratom=input("Molecule "+str(m2)+" ("+ms[m2]+"),("+str(matches[m1][atoms[m1][at1]][m2])+"): ")
                                if useratom == 'exit' or useratom == 'EXIT' or useratom == 'Exit':
                                    print("Exiting...")
                                    quit()
                                matches[m1][atoms[m1][at1]][m2]=[useratom]
                    print("")
            print("\n")
        printListDict(matches)
        quit()
    
    ## (iiJ) Make sure all core lists have the same number of atoms: Correct inconsistencies
    chkcores=[]
    for m1 in range(len(ms)):
        chkcores.append([])
        m1cores=list(cores[m1].keys())
        for m2 in range(len(ms)):
            chkcores[m1].append([])
            if m1 != m2:
                m2cores=list(cores[m2].keys())
                # generate list of m1 core matches from cores[m2] key-values
                m2matches=[]
                for at2 in m2cores:
                    m2matches.append(cores[m2][at2][m1][0])
                for at1 in m1cores:
                    if not (at1 in m2matches):
                        chkcores[m1][m2].append(at1)
                        # at1 = key in m1
                        line=deepcopy(cores[m1][at1])
                        line[m1]=[at1]
                        cores[m2][line[m2][0]]=line
                        cores[m2][line[m2][0]][m2]=[]
    
    # debug
    if debug:
        print("AFTER CORE CHKS")
        print("MATCHES")
        printListDict(matches)
        print("")
        print("CORE ATOMS")
        printListDict(cores)
        print("")
        print("")
    
    
    ## (5) Figure out the number of fragments in each molecule, pair them together between molecules,
    ## and remove redundancies
    fragbonds=deepcopy(bonds)         ## no H bonds, no core-core bonds
    for m1 in range(len(ms)):
        for at1 in range(len(atoms[m1])):
            for at2 in range(at1+1,len(atoms[m1])):
                if (atoms[m1][at1] in cores[m1].keys()) and (atoms[m1][at2] in cores[m1].keys()):
                        fragbonds[m1][at1][at2]=0
                        fragbonds[m1][at2][at1]=0
                if (xyzs[m1][atoms[m1][at1]]['Elem'] == 'H') or (xyzs[m1][atoms[m1][at2]]['Elem'] == 'H'):
                        fragbonds[m1][at1][at2]=0
                        fragbonds[m1][at2][at1]=0
        ##print("Molecule",m1)
        ##for at1 in range(len(atoms[m1])):
        ##    for at2 in range(len(atoms[m1])):
        ##        if at2 < at1:
        ##            print("  ",end='')
        ##        else:
        ##            print(" "+str(fragbonds[m1][at1][at2]),end='')
        ##    print("")
    
    # figure out the number of core-to-(not core) attachments
    nsites=[]
    Aatoms=[]
    for m in range(len(ms)):
        Aatoms.append({})
        chk=0
        for at1 in range(len(atoms[m])):
            if atoms[m][at1] in cores[m].keys():
                for at2 in range(len(atoms[m])):
                    if xyzs[m][atoms[m][at2]]['Elem'] != 'H':
                        if fragbonds[m][at1][at2] == 1:
                            chk+=1
                            if atoms[m][at1] in Aatoms[m].keys():
                                Aatoms[m][atoms[m][at1]].append(atoms[m][at2])
                            else:
                                Aatoms[m][atoms[m][at1]]=[atoms[m][at2]]
        nsites.append(chk)
    
    # initial nsites and Aatoms
    if debug:
        print("NSITES and AATOMS")
        print(nsites)
        print(Aatoms)
    
    
    # now take these sites and start expounding on them to list out all fragment atoms attached to them
    # (remember to look for rings (where one fragment branch joins another nsite/Aatom point)
    
    def GetIndex(atomname,molnum):
        """ given an atomname for a molecule number, return the atom index"""
        for atomindex in range(len(atoms[molnum])):
            if atoms[molnum][atomindex] == atomname:
                break
        return atomindex
    
    Fatoms=[]
    for m in range(len(ms)):
        Fatoms.append([])
        for k in Aatoms[m].keys():
            for fragatom in Aatoms[m][k]:
                Fatoms[m].append([])
                Fatoms[m][-1].append(fragatom)
                # find the atom index for fragatom
                at1 = GetIndex(fragatom,m)
                # enumerate all bonds to this atom (excluding core and H atoms)
                chk=1
                atlist=[]
                while chk > 0:
                    i=fragbonds[m][at1,:]
                    for j in range(len(i)):
                        if i[j] == 1:
                            if not (atoms[m][j] in cores[m].keys()) and not (atoms[m][j] in Fatoms[m][-1]):
                                atlist.append(j)
                    if atlist==[]:
                        chk=0
                    else:
                        at1=atlist.pop()
                        if not (atoms[m][at1] in Fatoms[m][-1]):
                            Fatoms[m][-1].append(atoms[m][at1])
        # check for redundant/identical fragments & merge if found
        droplist=[]   # list the indices to drop from Fatoms[m][-1]
        tsites=nsites[m]
        for k1 in range(tsites-1):
            for k2 in range(k1+1,tsites):
                chk=0
                tmpfrag=deepcopy(Fatoms[m][k2])
                for at1 in Fatoms[m][k1]:
                    for at2 in range(len(tmpfrag)):
                        if at1 == tmpfrag[at2]:
                            chk+=1
                            tmpfrag.pop(at2)
                            break
                if chk == len(Fatoms[m][k1]):
                    # then we have a match
                    if not (k2 in droplist):
                        nsites[m]=nsites[m]-1
                        droplist.append(k2)
        # merge found matches; modify: (i) Fatoms, (ii) Aatoms (as part of next routine)
        if len(droplist) > 0:
            tmp=[]
            droplist.sort(reverse=True) # to work highest index to lowest
            for i in droplist:
                tmp=Fatoms[m].pop(i)  # pop in reverse direction so we don't remove things we actually want
        # replace Aatom values with frag atom lists
        for frag in Fatoms[m]:
            droplist=[]
            for k in Aatoms[m].keys():
                for f in range(len(Aatoms[m][k])):
                    if Aatoms[m][k][f] in frag:
                        droplist.append(k)
            if len(droplist) > 1:
                # merge key_names and replace with new frag list
                newname=""
                for at in range(len(droplist)):
                    if at == 0:
                        newname+=droplist[at]
                    else:
                        newname+="+"+droplist[at]
    
                    if len(Aatoms[m][droplist[at]]) == 1:
                        Aatoms[m].pop(droplist[at]) # doesn't work if Aatoms[m][k] has more than one atom in it's nested list
                    else:
                        # try making a new Aatoms[m][at] list without the matching frag atom
                        for f in range(len(Aatoms[m][droplist[at]])):
                            if Aatoms[m][droplist[at]][f] in frag:
                                break
                        newlist=[]
                        for f2 in range(len(Aatoms[m][droplist[at]])):
                            if f != f2:
                                newlist.append(Aatoms[m][droplist[at]][f2])
                        Aatoms[m][droplist[at]]=newlist
                # check for newname already in Aatoms key
                if newname in Aatoms[m].keys():
                    print("ERROR: newname key already exists within Aatoms[m]")
                    print("       rework newname... and try again")
                    tmpNN=newname.split('+')
                    tmpnewname=''
                    for tnn in range(len(tmpNN)-1,0-1,-1):
                        tmpnewname+=tmpNN[tnn]+'+'
                    tmpnewname=tmpnewname[0:-1]
                    if tmpnewname in Aatoms[m].keys():
                        print("ERROR: tmpnewname",tmpnewname,"key already exists within Aatoms[m]")
                        quit()
                    Aatoms[m][tmpnewname] = deepcopy(frag)
                else:
                    Aatoms[m][newname] = deepcopy(frag)
            elif len(droplist) == 1:
                Aatoms[m][droplist[0]] = deepcopy(frag)
            else: #len(droplist) < 1:
                print("ERROR: frag mismatch between Fatoms list and Aatoms match")
                quit()
    
    # Do some quick checks:
    for m in range(len(ms)):
        if nsites[m] != len(Aatoms[m]) or nsites[m] != len(Fatoms[m]):
            print("ERROR: mismatch in the number of sites for molecule",m)
            print("MOLECULE",m)
            print("NSITES[M]",nsites[m])
            print("AATOMS[M]",Aatoms[m])
            print("FATOMS[M]",Fatoms[m])
            quit()
    
    
    if debug:
        # nsites, Aatoms, and Fatoms after sorting is complete
        print("")
        print("NSITES:",nsites,'\n')
        print("AATOMS:_______________________________")
        printListDict(Aatoms)
        print("FATOMS:_______________________________")
        printListList(Fatoms)
        print("")
    
    ## (6) Identify the reference ligand (either hardcode it as the first molecule, or pick the molecules with the smallest number of fragment atoms
    
    #refnum=0  # uncomment if you want to hardcode this value
    minfragatoms=-1
    for m in range(len(ms)):
        sumchk=0
        for k in Aatoms[m].keys():
            sumchk+=len(Aatoms[m][k])
        if minfragatoms == -1:
            minfragatoms=sumchk
            refnum=m
        else:
            if sumchk < minfragatoms:
                minfragatoms=sumchk
                refnum=m
    
    if debug:
        print("REFNUM =",refnum,"\n\n")
    
    
    ## Make sure all ligands have the same number of sites
    chk=0
    for m in range(len(ms)):
        if nsites[m] != nsites[refnum]:
            chk+=1
    if chk != 0:
        print("ERROR: Not all molecules have the same number of sites defined! Check and resubmit")
        quit()
    
    ## Order the fragments so that everything is consistent when we do redundant checks
    # use the refnum molecule to decide what fragment is first
    Ftemplate=[]
    for k in Aatoms[refnum].keys():
        Ftemplate.append(k)
    
    # generate the expected keys based off what's in cores, then check to make sure everything is correct
    def chkMergedKey(keytocheck):
        """ Check if a key is a merged key (name+name). Split into a list if yes, return value if not """
        tmp=keytocheck.split('+')
        if len(tmp) > 1:
            keyname = tmp
        else:
            keyname=[keytocheck]
        return keyname
    
    Forder=[]
    for m1 in range(len(ms)):
        Forder.append([])
        for temp in Ftemplate:
            tt=chkMergedKey(temp)
            if len(tt) > 1: # then it is a merged key
                tmp=""
                for t3 in range(len(tt)):
                    if m1 == refnum:
                        i=tt[t3]
                    else:
                        i=cores[refnum][tt[t3]][m1][0]
                    if t3 == 0:
                        tmp+=i
                    else:
                        tmp+="+"+i
                Forder[m1].append(tmp)
    
            else:           # it is a single value key
                if m1 == refnum:
                    Forder[m1].append(tt[0])
                else:
                    Forder[m1].append(cores[refnum][tt[0]][m1][0])
        # make sure no two keys are the same
        chk = 0
        for f1 in range(len(Forder[m1])):
            for f2 in range(len(Forder[m1])):
                if f1 != f2:
                    if Forder[m1][f1] == Forder[m1][f2]:
                        chk+=1
        if chk > 0:
            print("ERROR: Non-unique keys found linking core to fragment atoms!")
            quit()
        # make sure the generated key actually matches what's in Aatoms[m1]
        # and account for order differences
        chk = 0
        tmp=list(Aatoms[m1].keys())
        for f in Forder[m1]:
            f1=chkMergedKey(f)
            if len(f1) == 1:
                for f2 in range(len(tmp)):
                    if tmp[f2] == f:
                        chk+=1
                        tmp.pop(f2)
                        break
            else:
                # consider different orders; if found, update Aatoms keyname
                for f2 in range(len(tmp)):
                    f3=chkMergedKey(tmp[f2])
                    if len(f3) == len(f1):
                        chk2=0
                        for f4 in f1:
                            for f5 in range(len(f3)):
                                if f4 == f3[f5]:
                                    chk2+=1
                                    f3.pop(f5)
                                    break
                        if chk2 == len(f1):
                            chk+=1
                            kk=tmp.pop(f2)
                            break
                # update Aatoms key
                if kk != f:
                    Aatoms[m1][f]=Aatoms[m1][kk]
                    Aatoms[m1].pop(kk)
        if chk != len(Forder[m1]):
            print("ERROR: Mismatch between generated and actual Aatom keys!")
            print("Molecule",m1)
            print("AATOMS",Aatoms[m1].keys())
            print("FORDER",Forder[m1])
            quit()
    
    if debug:
        print("FORDER:")
        printListList(Forder)
    
    ## (7) Now look for redundancies in the fragments in each molecule so that we only print out a single fragment per site
    # refnum fragments are added by default
    
    # shortest distance function
    def findShortestDistance(atname1,atname2,molnum):
        """ Explore bonds to find the minimum distance between two atoms """
        at1=GetIndex(atname1,molnum)
        at2=GetIndex(atname2,molnum)
    
        ## at1 = "anchor atom"; start here and tree down until we find the atom we're looking for (at2)
        chk = 0
        atlist=[at1]
        while True:
            chk+=1
            bdlist=[]
            for i in range(len(atlist)):
                ii=bonds[molnum][atlist[i],:]
                for j in range(len(ii)):
                    if ii[j] == 1 and xyzs[molnum][atoms[molnum][j]]['Elem'] != 'H':
                        if not (j in bdlist):
                            bdlist.append(j)
            if at2 in bdlist:
                mindist = chk
                break
            else:
                atlist=deepcopy(bdlist)
    
        return mindist
    
    # use a compare function:
    def fragCompare(ufidx,ufkey,qidx,qkey):
        """ Compare two fragments & try to figure out if they are the same or not.
            Return "TRUE" if they match, otherwise, return "FALSE". """
        uflist=Aatoms[ufidx][ufkey]
        qlist=Aatoms[qidx][qkey]
    
        # reject if list lenghts are different
        if len(uflist) != len(qlist):
            return False
    
        # reject if lists have atoms with different bonding patterns or unequal #s of atoms with same bonding patterns
        #   i. calc bonding patterns for each atom in uflist and qlist
        ufpatt=[]
        for at in uflist:
            ufpatt.append(getBonded(GetIndex(at,ufidx),ufidx))
        qpatt=[]
        for at in qlist:
            qpatt.append(getBonded(GetIndex(at,qidx),qidx))
    
        #   ii. find shortest distance to a single core atom (first core atom in key if its a merged key)
        atname1=ufkey.split('+')[0]
        for atname2 in range(len(uflist)):
            mindist=findShortestDistance(atname1,uflist[atname2],ufidx)
            ufpatt[atname2].append(mindist)
        atname1=qkey.split('+')[0]
        for atname2 in range(len(qlist)):
            mindist=findShortestDistance(atname1,qlist[atname2],qidx)
            qpatt[atname2].append(mindist)
            
        qtemp=deepcopy(qpatt)
        
        #   iii. now compare types
        amatch=0
        for at1 in range(len(uflist)):
            for at2 in range(len(qlist)):
                if ufpatt[at1][0] == qtemp[at2][0] and ufpatt[at1][2] == qtemp[at2][2]: # atom type matches
                    bdsum=0
                    for bd1 in ufpatt[at1][1]:
                        if bd1 in qtemp[at2][1]:
                            bdsum+=1
                            #remove bd1 atomtype from qtemp[at2][1] to prevent duplicities
                            for i in range(len(qtemp[at2][1])):
                                if qtemp[at2][1][i] == bd1:
                                    qtemp[at2][1].pop(i)
                                    break
                    if bdsum == len(ufpatt[at1][1]): # then they match
                        amatch+=1
                        qtemp[at2][0]='MATCHED' # prevents duplicities
                        break
        
        if amatch != len(uflist):
            return False
    
        # else: they match
        return True
    
    
    UFrag = []  # list of Unique Fragment ms indices
    for site in range(nsites[refnum]):
        UFrag.append([refnum])
    
    for site in range(nsites[refnum]):
        skip=[refnum] # refnum is included in UFrag by default, so we want to skip it in our comparisons
        ifrag=-1
        while len(skip) != len(ms):
            for frag in range(ifrag+1,len(UFrag[site])):
                newfrag=-1
                for m1 in range(len(ms)):
                    if not (m1 in skip):
                        #if COMPARISON says the groups are the same, then add m1 to skip (b/c it's frag is already represented)
                        #otherwise (else), add first non-match to UFrag and do the comparison over again
                        #repeat until all ms indices are in skip (b/c all unique fragments are in UFrag[site]
                        matched=fragCompare(UFrag[site][frag],Forder[UFrag[site][frag]][site],m1,Forder[m1][site])
                        if matched: # matched == True
                            skip.append(m1)
                        else:
                            if newfrag==-1:
                                newfrag = m1
                                UFrag[site].append(newfrag)
                                skip.append(newfrag)
            ifrag=frag
        
    if debug:
        print("UFRAG")
        print(UFrag,"\n")
    
    ## (8) Print A formatted file of information to pass onto CRN (separate script)
    ## This also allows you to modify things by hand 
    #fp=open('MCS_for_MSLD.txt','w')
    fp=open(mcsout,'w')
    fp.write('# Maximum Common Substructure Search for Multisite Lambda Dynamics (JV 2022)\n')
    fp.write('# %d molecules processed\n\n' % (len(ms)))
    # (1) Print nsubs info
    fp.write("NSUBS")
    for site in range(nsites[refnum]):
        fp.write(" %d" % (len(UFrag[site])))
    fp.write("\n\n")
    fp.write("REFLIG %s\n\n" %(ms[refnum]))
    # (2) Print Core Atoms
    fp.write('CORE \n')
    for mol in range(len(ms)):
        fp.write("%s" % (ms[mol]))
        # print heavy atoms in the core
        for k in cores[refnum].keys():
            if mol == refnum:
                at=k
            else:
                at=cores[refnum][k][mol][0]
            fp.write(" %s" % (at))
            # print attached hydrogen atoms too
            at1=GetIndex(at,mol)
            for at2 in range(len(atoms[mol])):
                if bonds[mol][at1][at2] == 1:
                    if xyzs[mol][atoms[mol][at2]]['Elem'] == 'H':
                        fp.write(" %s" % (atoms[mol][at2]))
        fp.write("\n")
    fp.write("\n")
    # (3) Print Anchor Atoms
    fp.write('ANCHOR ATOMS\n')
    for mol in range(len(ms)):
        fp.write("%s" % (ms[mol]))
        for site in range(nsites[refnum]):
            aatom=Forder[mol][site]
            chk=0
            for c in range(len(aatom)):
                if aatom[c] == '+':
                    # we have a "merged" Aatom
                    chk=1
            if chk == 1:
                fp.write(" %s" % ("DUM"))
            else:
                fp.write(" %s" % (aatom))
        fp.write("\n")
    fp.write("\n")
    # (4) Print Fragments
    for site in range(nsites[refnum]):
        fp.write('SITE '+str(site+1)+' FRAGMENTS\n')
        for uf in UFrag[site]:
            fp.write("%s" % (ms[uf]))
            for at in Aatoms[uf][Forder[uf][site]]:
                fp.write(" %s" % (at))
                # also print attached hydrogen atoms
                at1=GetIndex(at,uf)
                for at2 in range(len(atoms[uf])):
                    if bonds[uf][at1][at2] == 1:
                        if xyzs[uf][atoms[uf][at2]]['Elem'] == 'H':
                            fp.write(" %s" % (atoms[uf][at2]))
            fp.write("\n")
        fp.write("\n")
    
    fp.write("END  \n")
    fp.close()
    
    
    # finished
    return ms[refnum]


In [11]:
MsldMCS(
    molfile="/home/raheelx/cphmd_walkthrough/mol2_aligned/mol_list.txt",
    mcsout="/home/raheelx/cphmd_walkthrough/mcs_results.txt",
    cutoff=0.8,
    debug=True
)


VERY BEGINNING
Molecule 0
O1 [[], ['O2', 'O3'], ['O1', 'O2', 'O3'], ['O1', 'O2', 'O3'], ['O2', 'O3'], ['O1', 'O2', 'O3']]
O2 [[], ['O2', 'O3'], ['O1', 'O2', 'O3'], ['O1', 'O2', 'O3'], ['O2', 'O3'], ['O1', 'O2', 'O3']]
O3 [[], ['O2', 'O3'], ['O1', 'O2', 'O3'], ['O1', 'O2', 'O3'], ['O2', 'O3'], ['O1', 'O2', 'O3']]
O4 [[], ['O4'], ['O4'], ['O4'], ['O4'], ['O4']]
O5 [[], ['O5'], [], ['O5'], [], []]
O6 [[], ['O6'], ['O6'], [], ['O6'], []]
N7 [[], ['N7'], ['N7'], ['N7'], ['N7'], []]
N8 [[], ['N8'], ['N8'], ['N8'], ['N8'], []]
N9 [[], ['N9'], ['N9'], [], ['N9'], []]
N10 [[], ['N10'], [], [], [], []]
C11 [[], ['C14'], ['C11', 'C14'], ['C11', 'C14'], ['C14'], ['C14']]
C12 [[], ['C12'], ['C12'], ['C12'], ['C12'], []]
C13 [[], ['C13'], ['C13'], ['C13'], ['C13'], ['C13']]
C14 [[], ['C14'], ['C11', 'C14'], ['C11', 'C14'], ['C14'], ['C14']]
C15 [[], ['C15'], ['C15'], ['C15'], ['C15'], []]
C16 [[], ['C16'], ['C16'], ['C16'], ['C16'], []]
C17 [[], ['C17'], ['C17'], ['C17'], ['C17'], []]
C18 [[], ['C18

'/home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_1'

In [14]:



# Paths
rtf_dir = "/home/raheelx/cphmd_walkthrough/cgenff_output"
mol2_base_dir = "/home/raheelx/cphmd_walkthrough/mol2_aligned"
mcs_results_file = "/home/raheelx/cphmd_walkthrough/MCS_out/mcs_results.txt"

# Parse each .rtf to extract charge and max penalty
lig_info = []

for fname in sorted(os.listdir(rtf_dir)):
    if fname.endswith(".rtf"):
        path = os.path.join(rtf_dir, fname)
        charge = 0
        penalties = []
        with open(path, 'r') as f:
            for line in f:
                if line.strip().startswith("RESI"):
                    parts = line.strip().split()
                    if len(parts) > 2:
                        try:
                            charge = float(parts[2])
                        except:
                            pass
                elif line.strip().startswith("ATOM"):
                    match = re.search(r'! *penalty *= *([0-9.]+)', line)
                    if match:
                        penalties.append(float(match.group(1)))
        lig_info.append({
            "lig_name": fname.replace(".rtf", ""),
            "charge": charge,
            "max_penalty": max(penalties) if penalties else 999.0  # assign large penalty if none found
        })

# Find the base state (charge = -1, lowest penalty)
middle_charge = sorted(set([info["charge"] for info in lig_info]))[1]  # middle of 3
base_candidates = [lig for lig in lig_info if lig["charge"] == middle_charge]
base_state = min(base_candidates, key=lambda x: x["max_penalty"])

print("✅ Selected base state:", base_state)

# Rewrite MCS results file with new REFLIG
with open(mcs_results_file, 'r') as f:
    lines = f.readlines()

with open(mcs_results_file, 'w') as f:
    for line in lines:
        if line.startswith("REFLIG"):
            new_ref = f"REFLIG {mol2_base_dir}/{base_state['lig_name']}\n"
            f.write(new_ref)
        else:
            f.write(line)


✅ Selected base state: {'lig_name': 'riboflavin_2', 'charge': -1.0, 'max_penalty': 999.0}


In [19]:
# Defining the CRN function

class CRN_Error(Exception):
    import sys
    sys.exit

def MsldCRN(mcsout,outdir,inFrag,AnCore,ChkQChange=True,verbose=False,debug=False,ll=True):
    """
    Using the information within mcsout, as well as previously supplied mol2 and toppar
    files, perform charge renormalization to generate MSLD suitable force field parameters

    Use ChkQChange=True to check for charge perturbations between different substituents
    Use verbose=True to get extra output
    Use debug=True to get LOTS of extra output
    Use ll=True to build the "large_lig.pdb" file needed for easy solvation with Lg_Solvate.sh
    """

    #################################################################
    ## Read the information from mcsout
    mols=[]      # list of file names
    cores=[]     # list of lists of core atom names (same indexing as mols)
    Aatoms=[]    # list of lists of anchor atom names (same indexing as mols)
    frags=[]     # list of lists of file names for the fragments
    Fatoms=[]    # list of lists of the fragment atom names
    fp=open(mcsout,'r')
    line=fp.readline()
    nsites=0
    while line:
        if line[0:4] == 'NSUB':
            nsubs=[int(x) for x in line.split()[1:]]
            line=fp.readline()
        if line[0:4] == 'REFL':
            reflig=line.split()[1]
            line=fp.readline()
        if line[0:4] == 'CORE':
            line=fp.readline()
            while (line != '\n') and (line[0:4] != 'ANCH'): 
                lns=line.split()
                mols.append(lns[0])
                cores.append(lns[1:])
                line=fp.readline()
        if line[0:4] == 'ANCH':
            line=fp.readline()
            while (line != '\n') and (line[0:4] != 'SITE'): 
                lns=line.split()
                if lns[0] != mols[len(Aatoms)]:
                    raise CRN_Error('File names do not match between CORE and ANCHOR ATOMS - check and resubmit')
                Aatoms.append(lns[1:])
                line=fp.readline()
        if line[0:4] == 'SITE':
            nsites+=1
            frags.append([])
            Fatoms.append([])
            line=fp.readline()
            while (line != '\n') and (line[0:4] != 'SITE') and (line[0:3] != 'END'):
                lns=line.split()
                frags[nsites-1].append(lns[0])
                Fatoms[nsites-1].append(lns[1:])
                line=fp.readline()
        if line[0:3] == 'END':
            break
        line=fp.readline()
    fp.close()
    if nsites != len(nsubs):
        raise CRN_Error('Mismatch on the # of sites - check and resubmit')

    # transform cores into a DataFrame (DF) with reflig column headers and mols indices
    refnum=0
    for mol in range(len(mols)):
        if mols[mol] == reflig:
            refnum=mol
            break
    coreheader=cores[refnum]
    cores=pd.DataFrame(cores,columns=coreheader,index=mols,dtype=str)

    if debug:
        print("nsubs = ",nsubs,'\n')
        print("reflig = ",reflig,'\n')
        print("cores =  ("+str(cores.shape[0])+" total)\n",cores,'\n')
        print("anchor atoms =  ("+str(len(Aatoms))+" total)\n",Aatoms,'\n')
        for site in range(nsites):
            print("Site "+str(site+1)+" Fragments =  ("+str(len(frags[site]))+" total)")
            for frag in range(len(frags[site])):
                print(frags[site][frag],Fatoms[site][frag])
            print("\n")
    #debug#

    #################################################################
    ## Read the information from each *.rtf
    rtfinfo=[]   # list of dictionary infomation
    rtfvers1=36  # rtf version info
    rtfvers2=1

    for mol in mols:
        #initialize with blank lists
        #rtfinfo.append({'NAME':mol,'QNET':0,'MASS':[],'ATOM':[],'BOND':[],'IMPR':[],'ATTYPE':{},'ATQ':{}})
        rtfinfo.append({'NAME':mol,'QNET':0,'MASS':[],'ATTYPE':{},'ATQ':{},'BOND':[],'IMPR':[],'LP':[]})
    
        fp=open(mol+'.rtf','r')
        line=fp.readline()
        if line[0] == '*' and mol == refnum:
            while line:  # look for the charmm ff version # (assuming it directly follows the title lines)
                line=fp.readline()
                if line[0] != '*':
                    lns=line.split()
                    rtfvers1=lns[0]
                    rtfvers2=lns[1]
                    break
        while line:
            if line[0:4] == 'MASS':
                while line[0:4] == 'MASS':
                    lns=line.split()
                    rtfinfo[-1]['MASS'].append(lns[2:])
                    line=fp.readline()
            if line[0:4] == 'RESI':
                rtfinfo[-1]['QNET']=line.split()[2]
                line=fp.readline()
            if line[0:4] == 'ATOM':
                while line[0:4] == 'ATOM':
                    lns=line.split()
                    #rtfinfo[-1]['ATOM'].append(lns[1:4])
                    rtfinfo[-1]['ATTYPE'][lns[1]]=lns[2]
                    rtfinfo[-1]['ATQ'][lns[1]]=float(lns[3])
                    line=fp.readline()
            if line[0:4] == 'BOND':
                while line[0:4] == 'BOND':
                    lns=line.split()
                    # account for multiple bond entries on a single line
                    bdnum = len(lns) - 1
                    if bdnum == 2:
                        rtfinfo[-1]['BOND'].append(lns[1:3])
                    else:
                        for i in range(int(bdnum/2)):
                            j=2*i+1
                            rtfinfo[-1]['BOND'].append(lns[j:(j+2)])
                    line=fp.readline()
            if line[0:4] == 'IMPR':
                while line[0:4] == 'IMPR':
                    lns=line.split()
                    rtfinfo[-1]['IMPR'].append(lns[1:5])
                    line=fp.readline()
            if line[0:4] == 'LONE':
                while line[0:4] == 'LONE':
                    lns=line.split()
                    rtfinfo[-1]['LP'].append(lns[2:])
                    line=fp.readline()
            if line[0:3] == 'END':
                break
            line=fp.readline()

        fp.close()

    # Deduce if LPs go into the core or into a fragment
    coreLPs=[]
    for mol in range(len(mols)):
        coreLPs.append([])
        for lp in rtfinfo[mol]['LP']:
            #if lp[1] in coreheader:
            if lp[1] in list(cores.loc[mols[mol]][:].values):
                #add to the corelist DataFrame
                coreLPs[mol].append(lp[0])
            else: # it is in a fragment
                 # figure out which fragment lp[1] is in and then add lp[0] into the list of fragment atoms
                for site in range(nsites):
                    for frag in range(len(frags[site])):
                        if frags[site][frag] == rtfinfo[mol]['NAME']:
                            if lp[1] in Fatoms[site][frag]:
                                Fatoms[site][frag].append(lp[0])

    # Add Core LPs into cores
    coreLPs=pd.DataFrame(coreLPs,columns=coreLPs[refnum],index=mols,dtype=str)
    if not coreLPs.empty:
        for lp in range(coreLPs.shape[1]):
            cores[coreLPs.iloc[refnum][lp]]=coreLPs.iloc[:,lp]

    if debug:
        # print LP info
        if not coreLPs.empty:
            print("New cores with LP atoms:\n",cores,'\n')
        for mol in range(len(mols)):
            print(rtfinfo[mol]['NAME'],rtfinfo[mol]['QNET'])
            #for field in ['MASS','ATOM','BOND','IMPR','ATTYPE','ATQ']:
            for field in ['MASS','ATTYPE','ATQ','BOND','IMPR','LP']:
                print(" -- "+field+' -- ')
                print(rtfinfo[mol][field])
                print()
            print('----------')
    #debug#

    # for the reference ligand, identify H atoms bonded to heavy atoms (in the core)
    Hcore={}
    #for at in cores.loc[reflig][:]:
    for at in coreheader:
        Hcore[at]=[]
        for bd in rtfinfo[refnum]['BOND']:
            if at in bd:
                if at[0] != 'H' and bd[0][0] == 'H':
                    Hcore[at].append(bd[0])
                elif at[0] != 'H' and bd[1][0] == 'H':
                    Hcore[at].append(bd[1])
                # readable both ways (heavy atom <-> H atom)
                elif at[0] == 'H' and bd[0][0] != 'H':
                    Hcore[at].append(bd[0])
                elif at[0] == 'H' and bd[1][0] != 'H':
                    Hcore[at].append(bd[1])
                else:
                    pass
    if debug:
        print("H atom bonds in the core:")
        print(Hcore)
    #debug#


    #################################################################
    ## Account for inFrag and AnCore specifications (modifies cores)
    ## "AnCore" is really only useful to specify Aatoms to be left in
    ## the core - otherwise, they are (by default) moved onto the frags

    # check for correct formats for inFrag and AnCore
    crtfrm='['
    for site in range(nsites):
        crtfrm+='[]'
        if site != (nsites-1):
            crtfrm+=','
    crtfrm+=']'
    if len(inFrag) != nsites:
        # if it isn't right, print an error message and setup a default inFrag variable to continue with
        print("\ninFrag is not specified correctly!! It should look like this: "+crtfrm)
        print("A default (empty) value will be used, but this may not yield the results you want")
        inFrag=[]
        for site in range(nsites):
            inFrag.append([])
    if len(AnCore) != nsites:
        # if it isn't right, print an error message and setup a default inFrag variable to continue with
        print("\nAnCore is not specified correctly!! It should look like this: "+crtfrm)
        print("A default (empty) value will be used, but this may not yield the results you want")
        AnCore=[]
        for site in range(nsites):
            AnCore.append([])
    # create list of atoms to move out of the core (add in Aatoms by default)
    droplist=[]
    for at in range(len(Aatoms[refnum])):
        droplist.append([])
        if Aatoms[refnum][at] != 'DUM':
            droplist[-1].append(Aatoms[refnum][at])
    # add in any inFrag atoms 
    for site in range(nsites):
        if len(inFrag[site]) > 0:
            for at in inFrag[site]:
                if not (at in droplist[site]):
                    droplist[site].append(at)
    # remove any AnCore atoms
    for site in range(nsites):
        if len(AnCore[site]) > 0:
            for at in AnCore[site]:
                if at in droplist[site]:
                    droplist[site].remove(at)

    # add in any H atoms bonded to droplist atoms
    tmp=deepcopy(droplist)
    for site in range(nsites):
        for at in tmp[site]:
            if at[0:1] == 'H':
                pass # don't check for atoms bonded to a H
            else:
                if len(Hcore[at]) > 0:
                    for hat in Hcore[at]:
                        if hat in coreheader and not (hat in droplist[site]):
                            droplist[site].append(hat)

    # modify existing variable structures
    flatlist=[at for site in droplist for at in site]
    drops=cores[flatlist]               # DF of droplist atoms extracted from cores
    cores=cores.drop(columns=flatlist)  # cores is modified to remove droplist atoms
    coreheader=list(cores.columns.values)
    for site in range(nsites):
        for frag in range(len(frags[site])):
            for at in droplist[site]:
                Fatoms[site][frag].append(drops.loc[frags[site][frag]][at])

    if debug:
        print('\nlist of atoms to drop from the core:\n',droplist,'\n')
        print('atoms cut from the cores DF:\n',drops,'\n')
        print('modified fragments are:')
        for site in range(nsites):
            print("Site "+str(site+1)+" Fragments =  ("+str(len(frags[site]))+" total)")
            for frag in range(len(frags[site])):
                print(frags[site][frag],Fatoms[site][frag])
            print("\n")
        print("\n")
    #debug#


    #################################################################
    ## Perform Charge Renormalization (CRN)

    offset=0.000001  # the amount by which charges are renormalized
    dec=6            # the precision of final (written) charges
    Qcut=5.0         # provide a warning for large charge diffs in the core

    # check for charge perturbations
    qnet=float(rtfinfo[refnum]['QNET'])
    delQ=[]    # list of charge change boolean
    for mol in range(len(mols)):
        delQ.append(False) # Charge Change boolean
        if float(rtfinfo[mol]['QNET']) != float(qnet):
            print("We have a charge change between the reflig ("+mols[refnum]+' and '+mols[mol]+').',\
                  "ref_qnet = "+str(qnet),"; mol_qnet = "+rtfinfo[mol]['QNET'])
            if not ChkQChange:
                print("    ***  Recommended to turn ON the ChkQChange option !!  ***")
            delQ[mol]=True

    #(1) Gather and Average Core Charges 
    Qcore=pd.DataFrame(np.zeros((cores.shape[0]+2,cores.shape[1])),columns=coreheader,index=mols+['mean','stdev'],dtype=float)
    for mol in range(len(mols)):
        for at in range(len(cores.iloc[mol][:])):
            Qcore.iloc[mol][at]=rtfinfo[mol]['ATQ'][cores.iloc[mol][at]]
    for col in Qcore:
        Qcore.loc['mean',col]=float(str(np.around(np.average(Qcore.loc[mols,col]),decimals=dec)))
        Qcore.loc['stdev',col]=float(str(np.around(np.std(Qcore.loc[mols,col]),decimals=dec)))
    if debug:
        print(Qcore)
        print()
    #debug#

    #(2) Gather Fragment totals & CRN
    Qfrag=[]
    Qint=[]
    siteavg=[]
    if verbose:
        print("\nFragment Charge Differences Following Charge ReNormalization:")
    for site in range(nsites):
        sitesum=[]
        Qfrag.append([])
        for frag in range(len(frags[site])):
            # collect all the charges into Qfrag (list of lists of Series of frag atom charges)
            Qfrag[site].append(pd.Series(np.zeros(len(Fatoms[site][frag])),index=Fatoms[site][frag]))
            for mol in range(len(mols)):
                if mols[mol] == frags[site][frag]:
                    break
            for at in Fatoms[site][frag]:
                Qfrag[site][frag][at]=rtfinfo[mol]['ATQ'][at]
            # get the total charge of each fragment at each site, the mean, and start to renormalize charge
            sitesum.append(float(str(np.around(Qfrag[site][frag].sum(),decimals=dec))))

        if ChkQChange and verbose:
            print("Charge Change Chk:","old sitesum",sitesum)
        # check for charge changes; if found - remove the nearest int charge and crn
        QQ=[0 for q in sitesum]
        if ChkQChange:
            orig_sitesum=deepcopy(sitesum)
            molQs = [rtfinfo[mol]['QNET'] == rtfinfo[refnum]['QNET'] for mol in range(len(mols))]
            if not all(molQs):
                QQ=np.rint(sitesum) # round to nearest integer with numpy
                QQ=[int(q) for q in QQ]
                sitesum=[sitesum[q]-float(QQ[q]) for q in range(len(sitesum))]
                # get majority Qint
                Qint.append(float(str(np.around(np.rint(np.asarray(QQ).mean()),decimals=2))))
        if ChkQChange and verbose:
            print("Charge Change Chk:","Qs round to these integers",QQ)
            print("Charge Change Chk:","new sitesum",sitesum)
            print(" ")

        # avg the sitesums, and progressively CRN each fragment atom
        siteavg.append(float(str(np.around(np.asarray(sitesum).mean(),decimals=dec))))
        if verbose:
            print("Site "+str(site+1)+" Q(avg) = "+str(siteavg[site]))
        for frag in range(len(frags[site])):
            qdiff=float(str(np.around(siteavg[site]-sitesum[frag],decimals=dec)))
            foffset=offset
            if qdiff < 0.0:
                foffset=foffset*-1.0
            nsteps=int(np.around(qdiff/foffset,decimals=0)) # should be a positive int
            atom=0
            for step in range(nsteps):
                # cycle back to the beginning of the molecule
                if atom == len(Fatoms[site][frag]):
                    atom=0
                # if atom name starts with LP = skip it
                if Fatoms[site][frag][atom][0:2] == 'LP':
                    while Fatoms[site][frag][atom][0:2] == 'LP': # account for many LPs next to each other
                        atom+=1
                        if atom == len(Fatoms[site][frag]):
                            atom=0
                # add the offset to a non-LP atom
                Qfrag[site][frag][atom]+=foffset
                atom+=1
            # check that total frag charge matches siteavg[site]
            if ChkQChange:
                qchk=float(str(np.around(Qfrag[site][frag].sum()-float(QQ[frag]),decimals=dec)))
            else:
                qchk=float(str(np.around(Qfrag[site][frag].sum(),decimals=dec)))
            #print(frags[site][frag],qchk,str(qchk),str(qchk)=='-0.0')
            if str(qchk) == '-0.0':    
                qchk = float(0.0)   
            #print(frags[site][frag],qchk,str(qchk))
            if str(qchk) != str(siteavg[site]):
                raise CRN_Error('Error for fragment CRN for site '+str(site+1)+', frag = '+frags[site][frag]+
                ". Total charge ("+str(qchk)+") doesn't match the site average ("+str(siteavg[site])+") ")
            else:
                if verbose:
                    if sitesum[frag] == 0.0:
                        pdiff=float(str(np.around(qdiff*100.0,decimals=2)))
                    else:
                        pdiff=float(str(np.around(qdiff/sitesum[frag]*100.0,decimals=2)))
                    print("  "+frags[site][frag]+" Q(diff from avg) = "+str(qdiff)+
                          " Q(orig) = ",end='')
                    if ChkQChange:
                        print(str(orig_sitesum[frag]),end='')
                    else:
                        print(str(sitesum[frag]),end='')
                    print(" (a "+str(pdiff)+"% diff from orig charges)",end='')
                    if abs(pdiff) > Qcut:
                        print(" ** CHECK")
                    else:
                        print("")
        if verbose:
            print("")


    #if debug:
    #    for site in range(nsites):
    #    #for site in range(1):
    #        print('\nCharges for fragments at site '+str(site+1)+':')
    #        for frag in range(len(frags[site])):
    #        #for frag in range(1):
    #            print(frags[site][frag])
    #            print(Qfrag[site][frag])
    #            print()
    #        print()
    ##debug#


    #(3) CRN Core charges to neutralize ligands
    # extract current core (mean) charges
    QQ=pd.Series(Qcore.loc['mean'][:])
    # figure out the difference between the reflig's charge and current site charges
    sitesum=float(str(np.around(np.sum(siteavg),decimals=dec)))  # sum of all site charges
    intsum=float(str(np.around(np.sum(Qint),decimals=dec)))
    QQsum=float(str(np.around(QQ.sum(),decimals=dec)))      # sum of core (mean) charges
    tsum=float(str(np.around(QQsum+sitesum+intsum,decimals=dec)))  # total charge of msld ligand
    qdiff=float(str(np.around(float(rtfinfo[refnum]['QNET'])-tsum,decimals=dec)))

    coffset=offset
    if qdiff < 0.0:
        coffset=coffset*-1.0
    nsteps=int(np.around(qdiff/coffset,decimals=0)) # should be a positive int

    if debug: 
        print("ideal total charge = ",rtfinfo[refnum]['QNET'])
        print("site averages = ",siteavg)
        print("sitesum = ",sitesum)
        print("intsum = ",intsum)
        print("QQsum = ",QQsum)
        print("tsum = ",tsum)
        print("qdiff = ",qdiff)
        print("coffset = ",coffset)
        print("nsteps = ",nsteps)
        print()
    #debug#

    if debug:
        QQold=QQ.copy()
    ##debug#

    # do charge renormalization
    atom=0
    for step in range(nsteps):
        # cycle back to the beginning of the molecule
        if atom == QQ.shape[0]:
            atom=0
        # if atom name starts with LP = skip it
        if QQ.index[atom] == 'LP':
            while QQ.index[atom] == 'LP':
                atom+=1
                if atom == QQ.shape[0]:
                    atom=0
        # add the offset to a non-LP atom
        QQ[atom]+=coffset
        atom+=1
    # check that total core charges match the neg of sitesum (+qnet accounts for charged molecules)
    qchk=float(str(np.around(-1*QQ.sum(),decimals=dec)))
    sitechk=float(str(np.around(sitesum+intsum-qnet,decimals=dec)))
    if debug: 
        print("QCHK",qchk) 
        print("SiteCHK",sitechk) 
        print("Qnet",qnet) 
    if str(qchk) == '-0.0':
        qchk = 0.0
    if str(qchk) != str(sitechk):
        raise CRN_Error("Error for core CRN. Total charge ("+str(qchk*-1)+
                        ") doesn't neutralize site charge sums ("+str(sitesum)+") ")
    else:
        if verbose:
            print("Core Q(sum) = "+str(qchk*-1))
            if QQsum == 0.0:
                pdiff=0.0
            else:
                pdiff=float(str(np.around(qdiff/QQsum*100,decimals=2)))
            print("  Q(orig) = "+str(QQsum)+" (a "+str(pdiff)+"% diff from orig charges)",end='')
            if abs(pdiff) > Qcut:
                print(" ** CHECK")
            else:
                print("")
    if verbose:
        print("")

    if debug:
        print("CRN Core Charges:\nName,Old-Q,CRN-Q")
        for at in range(QQ.shape[0]):
            print(QQ.index[at],QQold[at],QQ[at])
    ##debug##
    
    #################################################################
    ## Check that the output directory exists - mkdir if not
    os.system('if [ ! -d '+outdir+' ]; then mkdir '+outdir+'; fi')


    #################################################################
    ## Read the information from each *.mol2 & write the *.pdb files
    ## and rename all atom names to a standardized format
    ## (only read the information you need:
    ##    - core from reflig/refnum
    ##    - frags from frags[site] files

    segid='LIG'     # segid for the ligand
    resname=segid   # residue name for the ligand

    def getElementSymbol(atomname):
        """ Read a Sybil atom type out of a mol2 file and return the atomic symbol """
        # assume that if there isn't a '.' that atomname is good as is (H, Cl, Br, etc)
        if atomname.find('.') == -1:
            newname=atomname
        # otherwise, figure out where the '.' is and use the letters before it
        else:
            newname=atomname.split('.')[0]
        return newname

    # read in the XYZ coords for the core
    coreXYZ={}
    coreEmt={} # store sybil atom types for better element deduction
    fp=open(reflig+'.mol2','r')
    line=fp.readline()
    while line:
        if line[0:13] == '@<TRIPOS>ATOM':
            line=fp.readline()
            while line[0:13] != '@<TRIPOS>BOND':
                lns=line.split()
                if lns[1] in coreheader:
                    coreXYZ[lns[1]]=lns[2:5]
                    coreEmt[lns[1]]=lns[5]
                line=fp.readline()
            break
        else:
            line=fp.readline()
    fp.close()
    # translate atom names into standardized format
    Ctrans={}  # translation dictionary between old (key) and new (value)
    if len(coreheader) > 200:
        raise CRN_Error("Can't handle more than 200 core atoms")
    pnum=[1,65,48]  # chr(65) == 'A' (ord('A') == 65); chr(48) = '0'
    for at in coreheader:
        # use function to get atomic symbol
        if at[0:2] == 'LP':
            newname='LP'
        else:
            newname=getElementSymbol(coreEmt[at])
        if len(newname) > 3:
            raise CRN_Error("Error in determining atomic symbol for atom "+at+" in the core")
        char=len(newname)
        if char == 1: # single char element
            if pnum[0] < 10:
                newat=newname+'00'+str(pnum[0])
            elif pnum[0] > 9 and pnum[0] < 100:
                newat=newname+'0'+str(pnum[0])
            else:
                newat=newname+str(pnum[0])
            pnum[0]+=1
        elif char == 2: # double char element
            if pnum[1] > 90:
                raise CRN_Error("Too many double char elements in core (max=26)")
            else:
                newat=newname+chr(pnum[2])+chr(pnum[1])
            pnum[1]+=1
        else:
            raise CRN_Error("Error in Core atom name translation")
        Ctrans[at]=newat
    # write core.pdb (atom order does not matter - but put H's after heavy atoms anyways)
    added=[]
    fp=open(outdir+'/core.pdb','w')
    row=0
    for at in coreheader:
        if at[0] == 'H':
            pass
        elif at[0] == 'L': # elif at[0:2] == 'LP':
            # no LP atoms added!
            pass
        else:
            if not (at in added):
                added.append(at)
                row+=1
                fp.write("ATOM  %5d %-4s %4s%5d    %8.3f%8.3f%8.3f%6.2f%6.2f      %-4s\n" % (\
                         row,Ctrans[at],segid,1,float(coreXYZ[at][0]),float(coreXYZ[at][1]),float(coreXYZ[at][2]),1,0,segid))
                # check for bonded H's
                if len(Hcore[at]) > 0:
                    for at2 in Hcore[at]:
                        added.append(at2)
                        row+=1
                        fp.write("ATOM  %5d %-4s %4s%5d    %8.3f%8.3f%8.3f%6.2f%6.2f      %-4s\n" % (\
                                 row,Ctrans[at2],segid,1,float(coreXYZ[at2][0]),float(coreXYZ[at2][1]),float(coreXYZ[at2][2]),1,0,segid))
    fp.write("TER\nEND")
    fp.close()

    # work on the fragments next
    Hfrag=[]
    Ftrans=[]
    for site in range(nsites):
        Hfrag.append([])
        Ftrans.append([])
        for frag in range(len(frags[site])):
            # read in the fragment XYZ coords
            fragXYZ={}
            fragEmt={} # store sybil atom types for better element deduction
            fp=open(frags[site][frag]+'.mol2','r')
            line=fp.readline()
            while line:
                if line[0:13] == '@<TRIPOS>ATOM':
                    line=fp.readline()
                    while line and line[0:13] != '@<TRIPOS>BOND':
                        lns=line.split()
                        if len(lns) < 2:
                            line = fp.readline()
                            continue
                        if lns[1] in Fatoms[site][frag]:
                            fragXYZ[lns[1]]=lns[2:5]
                            fragEmt[lns[1]]=lns[5]
                        line=fp.readline()
                    break
                else:
                    line=fp.readline()
            fp.close()
            # identify mol # that equals frag #
            for mol in range(len(mols)):
                if mols[mol] == frags[site][frag]:
                    break
            # identify H atoms bonded to heavy atoms (in each fragment)
            Hfrag[site].append({})
            for at in Fatoms[site][frag]:
                Hfrag[site][frag][at]=[]
                for bd in rtfinfo[mol]['BOND']:
                    if at in bd:
                        if at[0] != 'H' and bd[0][0] == 'H':
                            Hfrag[site][frag][at].append(bd[0])
                        elif at[0] != 'H' and bd[1][0] == 'H':
                            Hfrag[site][frag][at].append(bd[1])
                        # readable both ways (heavy atom <-> H atom)
                        elif at[0] == 'H' and bd[0][0] != 'H':
                            Hfrag[site][frag][at].append(bd[0])
                        elif at[0] == 'H' and bd[1][0] != 'H':
                            Hfrag[site][frag][at].append(bd[1])
                        else:
                            pass
            #if debug:
            #    if frag == 0 and site == 0:
            #        print(frags[site][frag]+" XYZ coordinates")
            #        print(fragXYZ)
            #        print("H atom bonds in "+frags[site][frag])
            #        print(Hfrag[site][frag])
            #        print()
            ##debug#

            # translate atom names into standardized format
            Ftrans[site].append({})
            for at in Fatoms[site][frag]:
                # use function to get atomic symbol
                if at[0:2] == 'LP':
                    newname='LP'
                else:
                    newname=getElementSymbol(fragEmt[at])
                if len(newname) > 3:
                    raise CRN_Error("Error in determining atomic symbol for atom "+at+" in "+frags[site][frag])
                char=len(newname)
                if char == 1: # single char element
                    if pnum[0] > 999:
                        raise CRN_Error("Too many single char elements in ligand (max=1000)")
                    if pnum[0] < 10:
                        newat=newname+'00'+str(pnum[0])
                    elif pnum[0] > 9 and pnum[0] < 100:
                        newat=newname+'0'+str(pnum[0])
                    else:
                        newat=newname+str(pnum[0])
                    pnum[0]+=1
                elif char == 2: # double char element
                    if pnum[1] > 90:
                        #reset pnum[1] back to 'A' and increment pnum[2] by 1
                        pnum[1]=65
                        pnum[2]+=1
                    if pnum[2] > 57:
                        raise CRN_Error("Too many double char elements in ligand (max=260)")
                    else:
                        newat=newname+chr(pnum[2])+chr(pnum[1])
                    pnum[1]+=1
                else:
                    raise CRN_Error("Error in Frag atom name translation")
                Ftrans[site][frag][at]=newat

            # write the site[site]_sub[frag].pdb file
            added=[]
            fp=open(outdir+'/site'+str(site+1)+'_sub'+str(frag+1)+'_frag.pdb','w')
            row=0
            for at in Fatoms[site][frag]:
                if at[0] == 'H':
                    pass # H's follow the heavy atom
                elif at[0] == 'L': # elif at[0:2] == 'LP':
                    # no LP atoms added!
                    pass
                else:
                    if not (at in added):
                        added.append(at)
                        row+=1
                        fp.write("ATOM  %5d %-4s %4s%5d    %8.3f%8.3f%8.3f%6.2f%6.2f      %-4s\n" % (\
                                 row,Ftrans[site][frag][at],segid,1,float(fragXYZ[at][0]),float(fragXYZ[at][1]),float(fragXYZ[at][2]),1,0,segid))
                        # check for bonded H's
                        if len(Hfrag[site][frag][at]) > 0:
                            for at2 in Hfrag[site][frag][at]:
                                added.append(at2)
                                row+=1
                                fp.write("ATOM  %5d %-4s %4s%5d    %8.3f%8.3f%8.3f%6.2f%6.2f      %-4s\n" % (\
                                         row,Ftrans[site][frag][at2],segid,1,float(fragXYZ[at2][0]),float(fragXYZ[at2][1]),float(fragXYZ[at2][2]),1,0,segid))
            fp.write("TER\nEND")
            fp.close()
 
    # write translation file for old to new atom names
    filename='translation.txt'
    fp=open(filename,'w')
    fp.write("Original Atom Name -> New Atom Name\n")
    fp.write("CORE\n")
    for at in coreheader:
        fp.write("%s %s\n" % (at,Ctrans[at]))
    fp.write("\n")
    # also do it for the fragments
    for site in range(nsites):
        for frag in range(len(frags[site])):
            fp.write("SITE %d %s (from %s)\n" % (site+1,'site'+str(site+1)+'_sub'+str(frag+1),frags[site][frag]))
            for at in Fatoms[site][frag]:
                fp.write("%s %s\n" % (at,Ftrans[site][frag][at]))
            fp.write("\n")
        fp.write("\n")
    fp.close()
    if verbose:
        print("\nOld to New Atom Name Translations found in: "+filename)
        print("MSLD Files Written Into: "+outdir)
        print()

    # make a "large_lig.pdb" file for easier solv_prep preparation
    if ll:
        fp=open(outdir+'/large_lig.pdb','w')
        # add the core
        ip=open(outdir+'/core.pdb','r')
        for line in ip:
            if line[0:4] == 'ATOM':
                fp.write(line)
        ip.close()
        # add the largest fragment at each site
        for site in range(nsites):
            maxnum=0
            maxfrag=1
            for frag in range(len(frags[site])):
                if len(Fatoms[site][frag]) > maxnum:
                    print("maxnum from",'site'+str(site+1)+'_sub'+str(frag+1)+'_frag.pdb')
                    maxnum=len(Fatoms[site][frag])
                    maxfrag=frag
            ip=open(outdir+'/site'+str(site+1)+'_sub'+str(maxfrag+1)+'_frag.pdb','r')
            for line in ip:
                if line[0:4] == 'ATOM':
                    fp.write(line)
            ip.close()
        fp.write("TER\nEND")
        fp.close()


    #################################################################
    ## Write *.rtf 

    # write core.rtf
    fp=open(outdir+'/core.rtf','w')
    fp.write('* ligand core rtf file generated with msld_py_prep for MSLD (JV,LC)\n')
    fp.write('* (core from %s)\n* \n' % (reflig))
    fp.write('  %d %d\n' % (rtfvers1,rtfvers2))

    lp=open(outdir+'/lpsites.inp','w')
    lp.write("* Load LP Site Definitions (if applicable)\n*\n\n")

    # add in mass statements if there are any and remove redundancies
    # (assumes consistant atom typing across the different molecules)

    # core mass statements first
    addlist=[]
    tmplist=[x[0] for x in rtfinfo[refnum]['MASS']]  # generate tmp list of all atom types with mass statements
    for at in coreheader:
        for type in range(len(tmplist)):
            if rtfinfo[refnum]['ATTYPE'][at] == tmplist[type]:
                if not (rtfinfo[refnum]['ATTYPE'][at] in addlist):
                    addlist.append(rtfinfo[refnum]['ATTYPE'][at])
                    fp.write("MASS -1 %s %s %s\n" % (rtfinfo[refnum]['MASS'][type][0],
                             rtfinfo[refnum]['MASS'][type][1],rtfinfo[refnum]['MASS'][type][2]))
    # frag mass statements second
    for site in range(nsites):
        for frag in range(len(frags[site])):
            # figure mol equiv for frag
            for mol in range(len(mols)):
                if mols[mol] == frags[site][frag]:
                    break
            tmplist=[x[0] for x in rtfinfo[mol]['MASS']]
            # loop over atoms in each frag
            for at in Fatoms[site][frag]:
                for type in range(len(tmplist)):
                    if rtfinfo[mol]['ATTYPE'][at] == tmplist[type]:
                        if not (rtfinfo[mol]['ATTYPE'][at] in addlist):
                            addlist.append(rtfinfo[mol]['ATTYPE'][at])
                            fp.write("MASS -1 %s %s %s\n" % (rtfinfo[mol]['MASS'][type][0],
                                     rtfinfo[mol]['MASS'][type][1],rtfinfo[mol]['MASS'][type][2]))
    # continue with core.rtf atom block
    fp.write("\n")
    fp.write("RESI  %s    %5.3f\n" % (segid,qchk*-1))  # Writes the core net charge only 
    fp.write("GROUP \n")  # Not subdivided (an exercise left to the user if needed)
    for at in coreheader:
        if at[0] == 'H':
            pass
        elif at[0] == 'L': # if at[0:2] == 'LP':
            # LP atoms added, but no Hcore checks
            #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ctrans[at],rtfinfo[refnum]['ATTYPE'][at],QQ.loc[at]))
            fp.write("ATOM %-4s %-6s %10.6f \n" % (Ctrans[at],rtfinfo[refnum]['ATTYPE'][at],QQ.loc[at]))
            # Figure out the "lonepair coli" data and print it to lpsites.inp
            for ln in rtfinfo[refnum]['LP']:
                if ln[0] == at:
                    lp.write("LONEPAIR COLI sele atom @ligseg @resnum %s end -\n" % (Ctrans[ln[0]]))
                    lp.write("              sele atom @ligseg @resnum %s end -\n" % (Ctrans[ln[1]]))
                    lp.write("              sele atom @ligseg @resnum %s end -\n" % (Ctrans[ln[2]]))
                    #lp.write("         DIST %s SCAL %s\ncoor shake\n\n" % (ln[4],ln[6])) # doesn't work with CGenFF
                    lp.write("         DIST %s SCAL %s\ncoor shake\n\n" % (ln[4],'0.00'))
        else:
            #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ctrans[at],rtfinfo[refnum]['ATTYPE'][at],QQ.loc[at]))
            fp.write("ATOM %-4s %-6s %10.6f \n" % (Ctrans[at],rtfinfo[refnum]['ATTYPE'][at],QQ.loc[at]))
            # check for bonded H's
            if len(Hcore[at]) > 0:
                for at2 in Hcore[at]:
                    #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ctrans[at2],rtfinfo[refnum]['ATTYPE'][at2],QQ.loc[at2]))
                    fp.write("ATOM %-4s %-6s %10.6f \n" % (Ctrans[at2],rtfinfo[refnum]['ATTYPE'][at2],QQ.loc[at2]))
    # core.rtf bond & impr lines
    for bd in rtfinfo[refnum]['BOND']:
        if (bd[0] in coreheader) and (bd[1] in coreheader):
            fp.write("BOND %-4s %-4s\n" % (Ctrans[bd[0]],Ctrans[bd[1]]))
    for impr in rtfinfo[refnum]['IMPR']:
        if (impr[0] in coreheader) and (impr[1] in coreheader) and \
           (impr[2] in coreheader) and (impr[3] in coreheader):
            fp.write("IMPR %-4s %-4s %-4s %-4s\n" % (Ctrans[impr[0]],Ctrans[impr[1]],\
                     Ctrans[impr[2]],Ctrans[impr[3]]))
    fp.write('PATCH FIRST NONE LAST NONE\n\nEND')
    fp.close()

    # write site[site]_sub[frag].rtf files
    for site in range(nsites):
        for frag in range(len(frags[site])):
            # figure out mol #
            for mol in range(len(mols)):
                if mols[mol] == frags[site][frag]:
                    break
            # write atom block
            fp=open(outdir+'/site'+str(site+1)+'_sub'+str(frag+1)+'_pres.rtf','w')
            fp.write('* fragment patch rtf file generated with py_prep for MSLD (JV,LC)\n')
            fp.write('* (fragment from %s)\n* \n' % (frags[site][frag]))
            fp.write('  %d %d\n\n' % (rtfvers1,rtfvers2))
            if ChkQChange:             # Calc frag net charge
                qsum=0.0
                for at in Fatoms[site][frag]:
                    qsum+=Qfrag[site][frag][at]
                qsum=float(str(np.around(qsum,decimals=dec)))
                fp.write('PRES p%d_%d    %5.3f\n' % (site+1,frag+1,qsum))
            else:
                fp.write('PRES p%d_%d    %5.3f\n' % (site+1,frag+1,siteavg[site]))
            fp.write('GROUP \n')
            for at in Fatoms[site][frag]:
                if at[0] == 'H':
                    pass
                elif at[0] == 'L': # if at[0:2] == 'LP':
                    # LP atoms added, but no Hfrag checks
                    #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ftrans[site][frag][at],rtfinfo[mol]['ATTYPE'][at],Qfrag[site][frag][at]))
                    fp.write("ATOM %-4s %-6s %10.6f \n" % (Ftrans[site][frag][at],rtfinfo[mol]['ATTYPE'][at],Qfrag[site][frag][at]))
                    # Figure out the "lonepair coli" data and print it to lpsites.inp
                    for ln in rtfinfo[mol]['LP']:
                        if ln[0] == at:
                            lp.write("LONEPAIR COLI sele atom @ligseg @resnum %s end -\n" % (Ftrans[site][frag][ln[0]]))
                            lp.write("              sele atom @ligseg @resnum %s end -\n" % (Ftrans[site][frag][ln[1]]))
                            lp.write("              sele atom @ligseg @resnum %s end -\n" % (Ftrans[site][frag][ln[2]]))
                            #lp.write("         DIST %s SCAL %s\ncoor shake\n\n" % (ln[4],ln[6])) # doesn't work with CGenFF
                            lp.write("         DIST %s SCAL %s\ncoor shake\n\n" % (ln[4],'0.00'))
                else:
                    #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ftrans[site][frag][at],rtfinfo[mol]['ATTYPE'][at],Qfrag[site][frag][at]))
                    fp.write("ATOM %-4s %-6s %10.6f \n" % (Ftrans[site][frag][at],rtfinfo[mol]['ATTYPE'][at],Qfrag[site][frag][at]))
                    # check for bonded H's
                    if len(Hfrag[site][frag][at]) > 0:
                        for at2 in Hfrag[site][frag][at]:
                            #fp.write("ATOM %-4s %-6s %9.5f \n" % (Ftrans[site][frag][at2],rtfinfo[mol]['ATTYPE'][at2],Qfrag[site][frag][at2]))
                            fp.write("ATOM %-4s %-6s %10.6f \n" % (Ftrans[site][frag][at2],rtfinfo[mol]['ATTYPE'][at2],Qfrag[site][frag][at2]))
            # write bond and impr lines (does NOT account for neighboring sites directly bonded to this site!)
            for bd in rtfinfo[mol]['BOND']:
                coreseries=cores.loc[mols[mol]][:]
                corelist=list(coreseries.values)
                if ((bd[0] in corelist) and (bd[1] in Fatoms[site][frag])) or \
                   ((bd[1] in corelist) and (bd[0] in Fatoms[site][frag])) or \
                   ((bd[0] in Fatoms[site][frag]) and (bd[1] in Fatoms[site][frag])):
                    # translate atom names then print
                    if bd[0] in corelist:
                        bdtmp=coreseries[coreseries.isin([bd[0]])].index[0]
                        at1 = Ctrans[bdtmp]
                    else:
                        at1 = Ftrans[site][frag][bd[0]]
                    if bd[1] in corelist:
                        bdtmp=coreseries[coreseries.isin([bd[1]])].index[0]
                        at2 = Ctrans[bdtmp]
                    else:
                        at2 = Ftrans[site][frag][bd[1]]
                    fp.write("BOND %-4s %-4s\n" % (at1,at2))
            for impr in rtfinfo[mol]['IMPR']:
                chk=0 # boolean for if we write it or not
                for at in impr:
                    # if at least one atom is a part of the fragment - then we want to print it
                    if at in Fatoms[site][frag]:
                        chk=1
                if chk:
                    # core atoms for this fragment specifically
                    coreseries=cores.loc[mols[mol]][:]
                    corelist=list(coreseries.values)
                    # chk to see if the impr spans two sites
                    crossite=[]
                    for at in impr:
                        if at == 'SKIP': 
                            break
                        if (not(at in corelist)) and (not(at in Fatoms[site][frag])):
                            chk=0   # at is in another site
                            if not (at in Aatoms[mol]):
                                impr.append('SKIP') # can't do crosssite IMPRs if the atoms aren't Aatoms
                            else:
                                # find 2nd site #
                                for site2 in range(nsites):
                                    if at == Aatoms[mol][site2]:
                                        crossite.append(site2)
                    if len(crossite) > 0:
                        crossite=list(set(crossite))
                        if len(crossite) > 1:
                            print(" !! Can't print IMPR line between three sites! (skipped)")
                            break
                    # write the impr line
                    if chk:
                        fp.write("IMPR")
                        for at in impr:
                            if at in corelist:
                                attmp=coreseries[coreseries.isin([at])].index[0]
                                fp.write(" %-4s" % (Ctrans[attmp]))
                            else:
                                fp.write(" %-4s" % (Ftrans[site][frag][at]))
                        fp.write("\n")
                    else: # cross site impropers written as comments
                        if impr[-1] == 'SKIP':
                            print(' !! Cannot print Cross Site IMPR between sites involving non-anchor fragment atoms')
                        else:
                            print(' ** Cross Site IMPR Found ('+frags[site][frag]+') ** Manually UNcomment as needed')
                            for frag2 in range(len(frags[crossite[0]])):
                                fp.write("!IMPR")
                                for at in impr:
                                    if at in corelist:
                                        attmp=coreseries[coreseries.isin([at])].index[0]
                                        fp.write(" %-4s" % (Ctrans[attmp]))
                                    elif at in Ftrans[site][frag]:
                                        fp.write(" %-4s" % (Ftrans[site][frag][at]))
                                    else: # it is in other site's fragment
                                        # figure out mol2 #
                                        for mol2 in range(len(mols)):
                                            if mols[mol2] == frags[crossite[0]][frag2]:
                                                break
                                        fp.write(" %-4s" % (Ftrans[crossite[0]][frag2][Aatoms[mol2][crossite[0]]]))
                                fp.write("\n")

            fp.write('\nEND')
            fp.close()
    lp.close()
    print()



    #################################################################
    ## Write CHARMM (prep) input script - full_ligand.prm not needed to write this file

    fp=open(outdir+'/nsubs','w')
    nblocks=0
    for site in range(nsites):
        fp.write("%d " % (len(frags[site])))
        nblocks+=len(frags[site])
    fp.close()
    fp=open(outdir+'/nblocks','w')
    fp.write("%d" % (nblocks))
    fp.close()
    fp=open(outdir+'/nreps','w')
    fp.write("1")
    fp.close()


    return

In [20]:
# Using the CRN code

mcsout = "/home/raheelx/cphmd_walkthrough/MCS_out/mcs_results.txt"
outdir = "/home/raheelx/cphmd_walkthrough/charge_renormalized_output"

# Map your ligands to their mol2 files
inFrag = {
    "riboflavin_1": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_1.mol2",
    "riboflavin_2": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_2.mol2",
    "riboflavin_3": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_3.mol2",
    "riboflavin_4": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_4.mol2",
    "riboflavin_5": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_5.mol2",
    "riboflavin_6": "/home/raheelx/cphmd_walkthrough/mol2_Epik/riboflavin_6.mol2",
}

# Map ligands to their CGenFF .str and .prm files
AnCore = {
    "riboflavin_1": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_1.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_1.prm",
    },
    "riboflavin_2": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_2.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_2.prm",
    },
    "riboflavin_3": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_3.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_3.prm",
    },
    "riboflavin_4": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_4.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_4.prm",
    },
    "riboflavin_5": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_5.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_5.prm",
    },
    "riboflavin_6": {
        "str": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_6.str",
        "prm": "/home/raheelx/cphmd_walkthrough/cgenff_output/riboflavin_6.prm",
    },
}

# Now call your function:
MsldCRN(mcsout=mcsout, outdir=outdir, inFrag=inFrag, AnCore=AnCore, verbose=True)



inFrag is not specified correctly!! It should look like this: [[]]
A default (empty) value will be used, but this may not yield the results you want

AnCore is not specified correctly!! It should look like this: [[]]
A default (empty) value will be used, but this may not yield the results you want
We have a charge change between the reflig (/home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_2 and /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_1). ref_qnet = -1.0 ; mol_qnet = 0.000
We have a charge change between the reflig (/home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_2 and /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_4). ref_qnet = -1.0 ; mol_qnet = 0.000
We have a charge change between the reflig (/home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_2 and /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_5). ref_qnet = -1.0 ; mol_qnet = -2.000
We have a charge change between the reflig (/home/raheelx/cphmd_walkthrough/mol2_aligned/ribofla

/tmp/ipykernel_2536256/3364270731.py:317: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Qcore.iloc[mol][at]=rtfinfo[mol]['ATQ'][cores.iloc[mol][at]]
/tmp/ipykernel_2536256/3364270731.py:317: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensur

  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_1 Q(diff from avg) = 0.011167 Q(orig) = -0.03 (a -37.22% diff from orig charges) ** CHECK
  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_2 Q(diff from avg) = 0.011167 Q(orig) = -1.03 (a -37.22% diff from orig charges) ** CHECK
  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_3 Q(diff from avg) = 0.011167 Q(orig) = -1.03 (a -37.22% diff from orig charges) ** CHECK
  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_4 Q(diff from avg) = 0.011167 Q(orig) = -0.03 (a -37.22% diff from orig charges) ** CHECK
  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_5 Q(diff from avg) = 0.011167 Q(orig) = -2.03 (a -37.22% diff from orig charges) ** CHECK
  /home/raheelx/cphmd_walkthrough/mol2_aligned/riboflavin_6 Q(diff from avg) = -0.055833 Q(orig) = 0.037 (a -150.9% diff from orig charges) ** CHECK

Core Q(sum) = 0.018833
  Q(orig) = 0.018833 (a 0.0% diff from orig charges)


Old to New Atom Name Translation

In [27]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
combine_rtf_no_rounding.py

Merge a core RTF (RESI LIG) and all *_pres.rtf files in one folder
into a single CHARMM topology without altering any charges.
"""

import os
from pathlib import Path
from typing import List


def combine_rtf_files(core_rtf_path: str,
                      patches_dir:   str,
                      output_rtf_path: str) -> None:
    END       = "END"
    combined: List[str] = []
    seen_core = False                    # let only the first RESI LIG through

    # ------------------ core ----------------------------------------------
    with open(core_rtf_path) as fh:
        for ln in fh:
            if ln.strip().upper() == END:
                continue                             # discard internal ENDs

            # keep first RESI LIG exactly as-is
            if not seen_core and ln.lstrip().startswith("RESI"):
                combined.append(ln)
                seen_core = True
                continue

            # skip any further duplicate RESI LIG blocks
            if seen_core and ln.lstrip().startswith("RESI") and " LIG " in ln:
                continue

            combined.append(ln)

    combined.append("\n")                # blank line before patches

    # ------------------ patches -------------------------------------------
    for fname in sorted(os.listdir(patches_dir)):
        if not fname.endswith(".rtf") or fname == Path(core_rtf_path).name:
            continue                     # ignore non-rtf and core itself

        lines = Path(patches_dir, fname).read_text().splitlines(keepends=True)

        # skip header comments / blank lines / "36 1"
        i0 = 0
        while i0 < len(lines) and (
            lines[i0].strip() == "" or
            lines[i0].lstrip().startswith("*") or
            lines[i0].strip() == "36 1"
        ):
            i0 += 1

        block = [ln for ln in lines[i0:] if ln.strip().upper() != END]
        while block and block[-1].strip() == "":
            block.pop()

        combined.append("\n")
        combined.extend(block)

    # ------------------ final END -----------------------------------------
    combined.append("\nEND\n")

    Path(output_rtf_path).parent.mkdir(parents=True, exist_ok=True)
    Path(output_rtf_path).write_text("".join(combined))
    print(f"✅  Combined RTF written to {output_rtf_path}")


# --------------------------------------------------------------------------
# Example usage – adjust the paths to your directories
# --------------------------------------------------------------------------
if __name__ == "__main__":
    core_rtf = "/home/raheelx/cphmd_walkthrough/charge_renormalized_output/core.rtf"
    patch_dir = "/home/raheelx/cphmd_walkthrough/charge_renormalized_output"
    out_file = "/home/raheelx/cphmd_walkthrough/riboflavin_combined_patch/combined_patches.rtf"

    combine_rtf_files(core_rtf, patch_dir, out_file)


✅  Combined RTF written to /home/raheelx/cphmd_walkthrough/riboflavin_combined_patch/combined_patches.rtf


In [28]:
# Combining the param files into combined prm file

prm_folder = '/home/raheelx/cphmd_walkthrough/cgenff_output'
combined_prm_path = os.path.join(prm_folder, 'ligand_combined.prm')

prm_files = [f for f in os.listdir(prm_folder) if f.endswith('.prm')]

with open(combined_prm_path, 'w') as outfile:
    for prm_file in prm_files:
        file_path = os.path.join(prm_folder, prm_file)
        with open(file_path, 'r') as infile:
            content = infile.read()
            outfile.write(f"! Start of {prm_file}\n")
            outfile.write(content)
            outfile.write(f"\n! End of {prm_file}\n\n")

print(f"Combined {len(prm_files)} .prm files into {combined_prm_path}")

Combined 6 .prm files into /home/raheelx/cphmd_walkthrough/cgenff_output/ligand_combined.prm


In [30]:
# Creating the CHARMM input script

# Define your file paths
rtf_path = "/home/raheelx/cphmd_walkthrough/riboflavin_combined_patch/combined_patches.rtf"
prm_path = "/home/raheelx/cphmd_walkthrough/cgenff_output/ligand_combined.prm"
pdb_path = "/home/raheelx/cphmd_walkthrough/charge_renormalized_output/large_lig.pdb"

# Create the CHARMM input script content
charmm_input = f"""
read rtf card name {rtf_path}
read param card name {prm_path}
read sequence pdb name {pdb_path}
read coor pdb name {pdb_path}

generate LIG setup
ic param
ic build

write psf card name LIG.psf
write coor pdb name LIG.pdb

stop
"""

# Write to file
with open("build_ligand.inp", "w") as f:
    f.write(charmm_input)

print("CHARMM input file 'build_ligand.inp' created!")


CHARMM input file 'build_ligand.inp' created!
